In [ ]:
Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("FraudCleaning")
    .getOrCreate()
)

In [ ]:
Схема

In [ ]:
schema = StructType([
    StructField("transaction_id", LongType(), True),
    StructField("tx_datetime", StringType(), True),
    StructField("customer_id", LongType(), True),
    StructField("terminal_id", LongType(), True),
    StructField("tx_amount", DoubleType(), True),
    StructField("tx_time_seconds", LongType(), True),
    StructField("tx_time_days", LongType(), True),
    StructField("tx_fraud", IntegerType(), True),
    StructField("tx_fraud_scenario", IntegerType(), True)
])

In [ ]:
Чтение данных

In [ ]:
df = (
    spark.read
    .schema(schema)
    .option("header", "false")
    .csv("hdfs:///user/data/fraud/*.txt")
)

rows_before = df.count()

print("Rows before cleaning:", rows_before)

In [ ]:
Удаление дубликатов

In [ ]:
df = df.dropDuplicates(["transaction_id"])

print(
    "Rows after duplicates removal:",
    df.count()
)

In [ ]:
Удаление пустых записей

In [ ]:
df = df.dropna(
    subset=[
        "transaction_id",
        "tx_datetime",
        "customer_id",
        "terminal_id",
        "tx_amount",
        "tx_fraud"
    ]
)

print(
    "Rows after NULL cleanup:",
    df.count()
)

In [ ]:
Конвертация даты

In [ ]:
df = (
    df.withColumn(
        "tx_datetime",
        to_timestamp("tx_datetime")
    )
)

In [ ]:
Удаление битых дат

In [ ]:
df = df.filter(
    col("tx_datetime").isNotNull()
)

print(
    "Rows after date validation:",
    df.count()
)

In [ ]:
Удаление отрицательных сумм

In [ ]:
df = df.filter(
    col("tx_amount") > 0
)

print(
    "Rows after amount cleanup:",
    df.count()
)

In [ ]:
Очистка №6. Проверка fraud-флага

In [ ]:
df = df.filter(
    col("tx_fraud").isin(0, 1)
)

print(
    "After fraud validation:",
    df.count()
)

In [ ]:
Проверка terminal_id

In [ ]:
df = df.filter(
    col("terminal_id") >= 0
)

In [ ]:
Проверка customer_id

In [ ]:
df = df.filter(
    col("customer_id") >= 0
)

In [ ]:
Проверка fraud_scenario

In [ ]:
df = df.filter(
    col("tx_fraud_scenario")
    .isin(0, 1, 2, 3)
)

In [ ]:
Поиск выбросов

In [ ]:
quantiles = df.approxQuantile(
    "tx_amount",
    [0.25, 0.75],
    0.01
)

q1 = quantiles[0]
q3 = quantiles[1]

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

print(lower, upper)

In [ ]:
Ограничение выбросов

In [ ]:
df = (
    df.withColumn(
        "tx_amount",
        when(
            col("tx_amount") > upper,
            upper
        )
        .otherwise(col("tx_amount"))
    )
)

In [ ]:
Финальная статистика

In [ ]:
rows_after = df.count()

print(
    f"Removed rows: {rows_before - rows_after}"
)

print(
    f"Remaining rows: {rows_after}"
)

df.describe().show()

In [ ]:
Запись в Object Storage

In [ ]:
OUTPUT_PATH = (
    "s3a://spark-bucket-ek/cleaned-data"
)

In [ ]:
(
    df.write
    .mode("overwrite")
    .parquet(OUTPUT_PATH)
)

In [ ]:
Проверка результата

In [ ]:
clean_df = spark.read.parquet(
    OUTPUT_PATH
)

clean_df.printSchema()

clean_df.show(5)

print(clean_df.count())